# Geison no Google Colab

Fluxo oficial para validar o Geison na `main`. O notebook só prepara ambiente, configuração e chamadas ao CLI `qpcr-pipeline`; a lógica científica fica no pacote.

Fluxo deste teste: `panel.proposal` → `ACTION_REQUIRED / PANEL_APPROVAL_REQUIRED` → revisão humana → `panel approve` → `frozen_manifest` → `--resume` → checkpoints e `report.html`.

Execute as células em ordem. Em sessão existente a preparação usa `git pull --ff-only origin main`.


In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Geison/.git ]; then
  git -C Geison checkout main
  git -C Geison pull --ff-only origin main
else
  git clone --branch main https://github.com/BrunoDCamargo/Geison.git
fi
python -m pip install -e /content/Geison
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cd-hit mafft primer3
printf 'Commit em teste: '
git -C /content/Geison rev-parse HEAD


In [ ]:
%cd /content/Geison
!qpcr-pipeline doctor


## NCBI

A aquisição ao vivo exige `NCBI_EMAIL`; `NCBI_API_KEY` é opcional. Ambos ficam apenas no ambiente da sessão.


In [ ]:
import os
from getpass import getpass
ncbi_email = input("NCBI e-mail: ").strip()
if not ncbi_email:
    raise ValueError("NCBI_EMAIL is required for live NCBI acquisition")
os.environ["NCBI_EMAIL"] = ncbi_email
ncbi_api_key = getpass("NCBI API key (optional): ").strip()
if ncbi_api_key:
    os.environ["NCBI_API_KEY"] = ncbi_api_key
else:
    os.environ.pop("NCBI_API_KEY", None)


## 1. Proposta, dry-run e gate

A configuração começa em `panel.proposal`. O dry-run não pode criar o `outdir`. A primeira execução real deve parar em `ACTION_REQUIRED / PANEL_APPROVAL_REQUIRED`, gravar `panel_proposal.yaml` e ainda não criar checkpoint de `input`.


In [ ]:
%%bash
set -euo pipefail
rm -rf /content/geison_run
mkdir -p /content/geison_run
cat > /content/geison_run/config.yaml <<'YAML'
target:
  name: SARS-CoV-2-example
input:
  ncbi:
    accessions: [NC_045512.2]
panel:
  proposal:
    target:
      name: SARS-CoV-2-example
      taxid: null
      mode: broad_detection
      subtype: null
      groups:
        - name: reference
          required: true
          dataset_roles: [DESIGN, CHALLENGE]
          reasons: [colab_validation]
          proposed_by: [manual]
          sequence_selection: []
    non_targets:
      - name: Related coronavirus example
        taxid: null
        criticality: IMPORTANT
        dataset_roles: [CHALLENGE]
        reasons: [specificity_context]
        proposed_by: [manual]
        sequence_selection: []
    diagnostic_context:
      syndrome: respiratory infection
      geography: global
      sample_type: respiratory specimen
      vector: null
alignment: {enabled: true, threads: 2}
conservation: {enabled: true, window_size: 100, step_size: 25}
YAML
cat /content/geison_run/config.yaml


In [ ]:
from pathlib import Path
import json, shlex, subprocess
output_dir = Path("/content/geison_run/output")
dry_command = "qpcr-pipeline run /content/geison_run/config.yaml --dry-run --outdir /content/geison_run/output"
dry = subprocess.run(shlex.split(dry_command), capture_output=True, text=True)
print(dry.stdout)
assert dry.returncode == 0
assert "Panel approval required before scientific execution." in dry.stdout
assert not output_dir.exists()
first_command = "qpcr-pipeline run /content/geison_run/config.yaml --outdir /content/geison_run/output"
first_run = subprocess.run(shlex.split(first_command), capture_output=True, text=True)
print(first_run.stdout)
assert first_run.returncode == 3
assert "PANEL_APPROVAL_REQUIRED" in first_run.stdout
proposal_path = output_dir / "panel_proposal.yaml"
assert proposal_path.is_file()
assert not (output_dir / ".checkpoints/input/manifest.json").exists()
run_manifest = json.loads((output_dir / "run_manifest.json").read_text())
assert run_manifest["status"] == "ACTION_REQUIRED"
assert run_manifest["action_required"]["code"] == "PANEL_APPROVAL_REQUIRED"
print(proposal_path.read_text())


## 2. Revisão e aprovação humana

Revise o `panel_proposal.yaml` impresso acima. Digite `APROVAR` somente se o painel fizer sentido. O comando `qpcr-pipeline panel approve` cria `approved_panel.json`.


In [ ]:
import json, shlex, subprocess
from pathlib import Path
confirmation = input("Digite APROVAR para congelar o painel: ").strip()
if confirmation != "APROVAR":
    raise RuntimeError("Panel approval cancelled by user")
approval_command = "qpcr-pipeline panel approve /content/geison_run/output/panel_proposal.yaml --output /content/geison_run/approved_panel.json"
approval = subprocess.run(shlex.split(approval_command), capture_output=True, text=True)
print(approval.stdout)
assert approval.returncode == 0
approved_path = Path("/content/geison_run/approved_panel.json")
approved = json.loads(approved_path.read_text())
assert approved["status"] == "APPROVED"
assert approved["approved_by_user"] is True
assert approved["proposal_sha256"].startswith("sha256:")


## 3. `frozen_manifest` e `--resume`

Após a aprovação, a configuração aponta para o manifesto congelado e retoma o mesmo `outdir`.


In [ ]:
%%bash
set -euo pipefail
cat > /content/geison_run/config-approved.yaml <<'YAML'
target:
  name: SARS-CoV-2-example
input:
  ncbi:
    accessions: [NC_045512.2]
panel:
  frozen_manifest: /content/geison_run/approved_panel.json
alignment: {enabled: true, threads: 2}
conservation: {enabled: true, window_size: 100, step_size: 25}
YAML
cat /content/geison_run/config-approved.yaml


In [ ]:
from pathlib import Path
import json, shlex, subprocess
resume_command = "qpcr-pipeline run /content/geison_run/config-approved.yaml --outdir /content/geison_run/output --resume"
resumed = subprocess.run(shlex.split(resume_command), capture_output=True, text=True)
print(resumed.stdout)
if resumed.stderr:
    print(resumed.stderr)
assert resumed.returncode == 0
output_dir = Path("/content/geison_run/output")
for path in [
    output_dir / "panel/approved_panel.json",
    output_dir / ".checkpoints/panel/manifest.json",
    output_dir / ".checkpoints/input/manifest.json",
    output_dir / "run_manifest.json",
]:
    assert path.is_file(), path
    print("OK", path)
run_manifest = json.loads((output_dir / "run_manifest.json").read_text())
print("panel_provenance:", json.dumps(run_manifest.get("panel_provenance"), indent=2))


## 4. `report.html`

O relatório interativo fica em `/content/geison_run/output/report.html`. Para retomar em outra sessão, preserve `config-approved.yaml`, `approved_panel.json` e o diretório completo de saída com os checkpoints usados por `--resume`.


In [ ]:
from pathlib import Path
import socket, subprocess, sys, time
from google.colab import output
report_dir = Path("/content/geison_run/output")
report_path = report_dir / "report.html"
if not report_path.is_file():
    raise FileNotFoundError(report_path)
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    probe.bind(("127.0.0.1", 0))
    report_port = probe.getsockname()[1]
report_server = subprocess.Popen([sys.executable, "-m", "http.server", str(report_port), "--bind", "127.0.0.1", "--directory", str(report_dir)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
deadline = time.time() + 5
while time.time() < deadline:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        if probe.connect_ex(("127.0.0.1", report_port)) == 0:
            break
    time.sleep(0.1)
else:
    report_server.terminate()
    raise RuntimeError("Could not start local server for report.html")
output.serve_kernel_port_as_iframe(report_port, path="/report.html", height=900)
